# Análisis de falla: el detector de partes sobre imágenes recortadas

**Módulo 2 — Segmentación (Sprint 2).** Este notebook documenta y *mide* una limitación
concreta del checkpoint `models/checkpoints/car_parts/v1/`: funciona bien sobre fotografías
del vehículo completo, pero se degrada de forma severa cuando la entrada es un *close-up*
recortado alrededor del daño — que es justamente el encuadre típico de un peritaje real.

La hipótesis es que **no se trata de un checkpoint defectuoso ni de falta de resolución, sino
de una brecha de dominio en el encuadre**: el modelo se entrenó con 333 imágenes del dataset
*Car parts coco-segmentation*, casi todas de vehículo completo en vista 3/4 sobre fondo de
estudio. Nunca vio un recorte.

El notebook contrasta cuatro condiciones:

| § | Condición | Qué demuestra |
|---|---|---|
| 1 | Imagen *in-domain* (split test de Roboflow) | El modelo sí funciona: es el control positivo |
| 2 | `car-test.jpg` — close-up recortado | El caso de falla que motiva el análisis |
| 3 | `car-test4.jpg` — vehículo completo | Misma calidad de foto, encuadre distinto, funciona |
| 4 | Recorte progresivo + control de resolución | Aísla la variable y produce una curva de degradación |

> **Requiere el kernel `venv_cuda`.** Ejecutar desde `notebooks/`.

In [ ]:
import os
import sys

# Los notebooks resuelven rutas contra la raíz del proyecto (el padre de notebooks/),
# no contra el cwd de Jupyter.
PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision.transforms import functional as F

from src.detection.common.predict import load_checkpoint
from src.detection.common.visualize import draw_detections

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PARTS_CKPT = os.path.join(PROJECT_ROOT, "models", "checkpoints", "car_parts", "v1", "best_model.pth")
DAMAGE_CKPT = os.path.join(PROJECT_ROOT, "models", "checkpoints", "damage", "v1", "best_model.pth")
DEMO_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "final_test")

parts_model, parts_categories = load_checkpoint(PARTS_CKPT, DEVICE)
damage_model, damage_categories = load_checkpoint(DAMAGE_CKPT, DEVICE)

print(f"Dispositivo: {DEVICE}")
print(f"Partes : {len(parts_categories)} clases de primer plano")
print(f"Danios : {len(damage_categories)} clases de primer plano")

In [ ]:
SCORE_THRESHOLD = 0.5


@torch.no_grad()
def predict(model, image_bgr, score_threshold=SCORE_THRESHOLD):
    """Corre un detector sobre una imagen BGR y filtra por confianza.

    Devuelve ``(boxes, labels, scores, masks)``; ``masks`` es None para Faster R-CNN.
    """
    tensor = F.to_tensor(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)).to(DEVICE)
    output = model([tensor])[0]
    keep = output["scores"] >= score_threshold
    masks = output["masks"][keep].cpu().numpy() if "masks" in output else None
    return (output["boxes"][keep].cpu().numpy(),
            output["labels"][keep].cpu().numpy(),
            output["scores"][keep].cpu().numpy(),
            masks)


def annotate(image_bgr, model, categories, score_threshold=SCORE_THRESHOLD):
    """Dibuja las predicciones de un detector sobre una copia de la imagen."""
    boxes, labels, scores, masks = predict(model, image_bgr, score_threshold)
    return draw_detections(image_bgr.copy(), boxes, labels, scores, categories, masks=masks)


def show(images, titles, figsize_per_image=(7, 5)):
    """Muestra una fila de imagenes BGR con sus titulos."""
    fig, axes = plt.subplots(1, len(images),
                             figsize=(figsize_per_image[0] * len(images), figsize_per_image[1]))
    for ax, image, title in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def summarize(model, categories, image_bgr, score_threshold=SCORE_THRESHOLD):
    """Imprime las detecciones ordenadas por confianza."""
    boxes, labels, scores, _ = predict(model, image_bgr, score_threshold)
    print(f"{len(boxes)} detecciones >= {score_threshold}")
    for i in np.argsort(-scores):
        print(f"  {categories.get(int(labels[i]), labels[i]):<20s} {scores[i]:.2f}")
    return boxes, labels, scores


def load(path, long_side=None):
    """Lee una imagen BGR y opcionalmente la escala a un lado largo dado."""
    image = cv2.imread(path)
    if image is None:
        raise FileNotFoundError(path)
    if long_side is not None:
        h, w = image.shape[:2]
        scale = long_side / max(h, w)
        image = cv2.resize(image, (int(w * scale), int(h * scale)))
    return image

---
## 1. Control positivo — imagen del dominio de entrenamiento

Antes de acusar al modelo hay que descartar que simplemente esté mal entrenado. Corremos el
mismo checkpoint sobre una imagen del **split `test` del propio dataset de partes**, que nunca
se usó para entrenar pero comparte el encuadre del set: vehículo completo, vista 3/4, fondo
de estudio.

In [ ]:
ROBOFLOW_TEST = os.path.join(PROJECT_ROOT, "data", "raw", "Car parts coco-segmentation", "test")

candidates = sorted(f for f in os.listdir(ROBOFLOW_TEST) if f.lower().endswith((".jpg", ".jpeg", ".png")))
in_domain_image = load(os.path.join(ROBOFLOW_TEST, candidates[1]), long_side=800)

show([annotate(in_domain_image, parts_model, parts_categories)],
     ["Control positivo: imagen in-domain (split test de Roboflow)"], figsize_per_image=(11, 8))

_ = summarize(parts_model, parts_categories, in_domain_image)

El modelo recupera una descripción **densa y coherente** del vehículo, con la lateralidad
correcta (`left_front_door`, `right_quarter_glass`, `side_mirror`, `radiator`, `windshield`…).
El checkpoint está sano. Cualquier falla posterior hay que atribuirla a la **entrada**, no a
los pesos.

---
## 2. El caso de falla — `car-test.jpg`

`car-test.jpg` es un *close-up* del costado trasero de un sedán con un golpe profundo en el
cuarto trasero. Es exactamente el tipo de foto que produciría un usuario documentando un daño:
encuadra el daño, no el coche.

Corremos **los dos módulos** sobre ella para ver el contraste.

In [ ]:
close_up = load(os.path.join(DEMO_DIR, "car-test.jpg"), long_side=900)

show([annotate(close_up, parts_model, parts_categories),
      annotate(close_up, damage_model, damage_categories)],
     ["Modulo 2 - Partes (falla)", "Modulo 3 - Danios (correcto)"])

print("=== PARTES ===")
summarize(parts_model, parts_categories, close_up)
print("\n=== DANIOS ===")
_ = summarize(damage_model, damage_categories, close_up)

El contraste es el punto central de este notebook:

- **El detector de daños acierta**: `dent` con confianza ~1.00 y una máscara que sigue el
  pliegue de la lámina con precisión.
- **El detector de partes falla**: sobreviven apenas 4 cajas al umbral de 0.5, contra las 29
  de la sección anterior. Etiqueta `trunk` (cajuela) sobre la ventanilla de la puerta trasera,
  `front_fender` (salpicadera delantera) sobre el pilar C, y — lo más grave — **`hood` (cofre)
  justo encima del golpe**, en una fotografía donde el cofre ni siquiera aparece. Solo
  `back_bumper` cae en una zona geográficamente plausible, y aun así con la caja muy
  desbordada.

### Por qué esto importa más allá del módulo 2

El objetivo de la tesis es reportar *qué se dañó y dónde*. Ese reporte se construye cruzando la
máscara del daño con las cajas de las partes. Con estas predicciones, el cruce afirmaría
**«golpe en el cofre»** sobre una foto que es puro costado trasero: la falla del módulo 2
**se propaga** y contamina la salida final aunque el módulo de daños sea correcto.

---
## 3. Contraste — `car-test4.jpg`, mismo modelo, vehículo completo

Para descartar que el problema sea la calidad de la fotografía, corremos el **mismo checkpoint**
sobre otra imagen de la misma carpeta, de resolución comparable, pero que muestra el
**vehículo completo**.

In [ ]:
full_car = load(os.path.join(DEMO_DIR, "car-test4.jpg"), long_side=900)

show([annotate(full_car, parts_model, parts_categories)],
     ["car-test4.jpg - vehiculo completo"], figsize_per_image=(12, 8))

_ = summarize(parts_model, parts_categories, full_car)

Sobre el vehículo completo el modelo vuelve a comportarse bien: recupera ruedas, llantas,
cofre, parabrisas, cristales, faros y manija con confianzas altas.

Mismo modelo, misma carpeta, resolución equivalente. **La única variable que cambió es el
encuadre.**

> *Nota sobre el material:* `car-test4.jpg` conserva impreso el patrón de damero de un fondo
> transparente. Es una textura artificial que el modelo nunca vio en entrenamiento; no impide
> el análisis, pero conviene tenerlo presente al leer las confianzas.

---
## 4. Experimento — recorte progresivo

Las secciones anteriores comparan imágenes *distintas*, así que todavía se les puede objetar
que algo más cambió entre ellas. Este experimento elimina esa objeción: parte de
`car-test4.jpg` — donde el modelo **sí** funciona — y la recorta progresivamente hacia el
daño, manteniendo todo lo demás constante.

Si la hipótesis del encuadre es correcta, el desempeño debe caer conforme desaparece el
contexto del vehículo, hasta reproducir la falla de `car-test.jpg`.

In [ ]:
# Centramos los recortes en el danio: es hacia donde encuadraria un usuario real.
# El detector de danios apenas lo alcanza a esta escala, asi que bajamos el umbral
# solo para localizarlo (no para reportarlo).
d_boxes, _, d_scores, _ = predict(damage_model, full_car, score_threshold=0.05)
h, w = full_car.shape[:2]
if len(d_boxes):
    x1, y1, x2, y2 = d_boxes[int(np.argmax(d_scores))]
    center = (int((x1 + x2) / 2), int((y1 + y2) / 2))
else:
    center = (w // 2, h // 2)
print(f"Centro del recorte (centroide del danio): {center}")


def crop_fraction(image, fraction, center):
    """Recorta una ventana centrada que conserva ``fraction`` de cada dimension."""
    h, w = image.shape[:2]
    cw, ch = int(w * fraction), int(h * fraction)
    x1 = max(0, min(w - cw, center[0] - cw // 2))
    y1 = max(0, min(h - ch, center[1] - ch // 2))
    return image[y1:y1 + ch, x1:x1 + cw]


FRACTIONS = [1.0, 0.8, 0.6, 0.45, 0.3, 0.2]

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
crop_results = []

for ax, fraction in zip(axes.ravel(), FRACTIONS):
    crop = crop_fraction(full_car, fraction, center)
    boxes, labels, scores, _ = predict(parts_model, crop)
    crop_results.append({
        "fraction": fraction,
        "size": f"{crop.shape[1]}x{crop.shape[0]}",
        "n_detections": len(boxes),
        "mean_score": float(scores.mean()) if len(scores) else 0.0,
        "labels": sorted({parts_categories.get(int(i), str(i)) for i in labels}),
    })
    ax.imshow(cv2.cvtColor(annotate(crop, parts_model, parts_categories), cv2.COLOR_BGR2RGB))
    ax.set_title(f"{int(fraction * 100)}% del encuadre - {len(boxes)} partes >= {SCORE_THRESHOLD}",
                 fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.show()

for r in crop_results:
    print(f"{int(r['fraction'] * 100):3d}%  {r['size']:>9s}  {r['n_detections']:2d} partes  "
          f"score medio {r['mean_score']:.2f}  {r['labels']}")

### El control de resolución

Al recortar también se reduce el número de píxeles, así que un escéptico podría atribuir la
caída a la resolución y no al encuadre. Lo separamos con una condición de control: reducir la
imagen **completa** exactamente al mismo tamaño en píxeles que cada recorte.

- Si manda la **resolución**, ambas curvas deben caer igual.
- Si manda el **encuadre**, solo debe caer la del recorte.

In [ ]:
sizes, n_cropped, n_scaled = [], [], []

for fraction in FRACTIONS:
    cropped = crop_fraction(full_car, fraction, center)
    ch, cw = cropped.shape[:2]
    # Misma cuenta de pixeles, pero conservando el vehiculo completo.
    scaled = cv2.resize(full_car, (cw, ch), interpolation=cv2.INTER_AREA)

    n_cropped.append(len(predict(parts_model, cropped)[0]))
    n_scaled.append(len(predict(parts_model, scaled)[0]))
    sizes.append(f"{cw}x{ch}")

print(f"{'tamanio':>10s}  {'recortado':>10s}  {'reescalado':>11s}")
for size, a, b in zip(sizes, n_cropped, n_scaled):
    print(f"{size:>10s}  {a:>10d}  {b:>11d}")

x = [f * 100 for f in FRACTIONS]
plt.figure(figsize=(9, 5))
plt.plot(x, n_cropped, "o-", linewidth=2, label="Recortado (se pierde contexto)")
plt.plot(x, n_scaled, "s--", linewidth=2, label="Reescalado (vehiculo completo, mismos pixeles)")
plt.gca().invert_xaxis()
plt.xlabel("Porcentaje del encuadre original conservado (%)")
plt.ylabel(f"Partes detectadas (score >= {SCORE_THRESHOLD})")
plt.title("Degradacion del detector de partes: encuadre vs. resolucion")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
## 5. Diagnóstico

**El resultado es concluyente.** La curva del recorte cae de forma monótona —
18 → 14 → 10 → 8 → 2 → 0 partes — mientras que la del reescalado, con el **mismo número de
píxeles** en cada punto, se sostiene: 18 → 17 → 17 → 15 → 13 → 10. En el extremo (180×120 px)
el recorte no detecta **nada** y el vehículo completo del mismo tamaño todavía recupera **10
partes**. La resolución no explica la caída; el encuadre sí.

Más revelador todavía: al 30% del encuadre las dos únicas detecciones que sobreviven son
`hood 0.71` y `hood 0.65`, **sobre una puerta lateral** — exactamente el mismo error que comete
el modelo sobre `car-test.jpg`. El experimento controlado **reproduce la falla observada**, lo
que indica que ambas comparten la misma causa.

### Causa

Faster R-CNN clasifica cada región propuesta apoyándose en el contexto que la ResNet50-FPN
codifica alrededor de ella. Con el vehículo completo visible, una lámina ambigua se resuelve
por su posición relativa: si está detrás de la puerta trasera es un cuarto trasero, no un
cofre. Al recortar, esa evidencia desaparece y el clasificador cae de vuelta hacia las clases
de panel amplio y liso que dominan su set de entrenamiento, entre ellas `hood`.

El dataset lo agrava: son **333 imágenes de entrenamiento**, casi todas de vehículo completo
en vista 3/4 sobre fondo de estudio. El modelo nunca vio un recorte.

### Consecuencias para el diseño del prototipo

1. **El cruce daño→parte no es confiable sobre close-ups.** Es la salida que el prototipo
   pretende entregar, así que esta limitación bloquea el reporte final, no solo el módulo 2.
2. **Hay que restringir la entrada o robustecer el modelo.** Cualquiera de las dos es una
   decisión de diseño explícita, pero tiene que tomarse.

### Acciones propuestas para el Sprint 2

- **Reconsiderar los recortes aleatorios en el aumento de datos.** Hoy
  `src/detection/common/transforms.py` los excluye deliberadamente y solo aplica volteo
  horizontal más *jitter* de color. Este notebook es el argumento empírico para reevaluar esa
  decisión: un recorte aleatorio moderado atacaría directamente la brecha medida aquí.
- **Ampliar el set con close-ups anotados**, que es el encuadre real de uso.
- **Validar sobre recortes, no solo sobre el split de Roboflow.** La métrica actual
  (AP@0.5:0.95 = 0.5389 en test) se mide íntegramente *in-domain* y por lo tanto **no captura
  esta falla en absoluto**: es optimista respecto al desempeño en operación.
- **Mientras tanto, exigir vehículo completo en la captura**, o detectar el encuadre y avisar
  al usuario cuando la foto esté demasiado cerrada.